# 简单对话 Agent：让对话“记得住”上下文

## 概述

这一节我们实现一个能保持上下文的对话 Agent：**同一个会话里连续对话时，模型能记住你上一轮说了什么**。

## 动机

很多最小聊天示例默认是“无状态”的：每次只喂最新一句话，导致前后不连贯。这里我们用“会话级短期记忆”把多轮对话串起来。

## 关键组件

- **Language Model**：负责生成回复
- **Prompt / Messages**：对话输入由一串 messages 组成
- **History Manager（短期记忆）**：用 `checkpointer` 把同一会话的 messages 持久化在 thread state 里
- **Session Identifier（会话标识）**：用 `configurable.thread_id` 标识不同会话

## 方法细节

- 用 LangGraph 定义一个最小图：输入 `messages` → 调用模型 → 把模型回复追加回 `messages`
- 编译时挂上 `MemorySaver()`，让同一个 `thread_id` 的多次调用自动累积历史


## 0) 导入依赖并加载环境变量

In [1]:
from __future__ import annotations

import os
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages


load_dotenv("../.env")

True

## 1) 初始化模型 + 定义 LangGraph（带短期记忆）

这里我们定义一个最小 LangGraph：

- state 里只有一个字段：`messages`
- 图里只有一个节点：把 `messages` 喂给模型，拿到 `AIMessage`，再追加回 `messages`
- 编译时传入 `checkpointer=MemorySaver()`，让同一个 `thread_id` 的多次调用自动累积历史


In [2]:
llm = ChatOpenAI(
    model="deepseek-v4-flash-0731",
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
    base_url=os.environ.get("DASHSCOPE_BASE_URL"),
    temperature=0,
)


class State(TypedDict):
    # 使用 reducer 来“追加消息”，而不是覆盖
    messages: Annotated[list, add_messages]


def call_model(state: State) -> dict:
    # 核心：把当前 messages 喂给模型，再把模型回复追加回去
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


builder = StateGraph(State)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")
builder.add_edge("call_model", END)

# 短期记忆：同一个 thread_id 会把 state（含 messages）累积起来
checkpointer = MemorySaver()
app = builder.compile(checkpointer=checkpointer)

## 2) 同一个会话：连续两轮对话

In [3]:
session_id = "user_123"
config = {"configurable": {"thread_id": session_id}}


r1 = app.invoke(
    {"messages": [{"role": "user", "content": "Hello! How are you?"}]},
    config=config,
)
print("AI:", r1["messages"][-1].content)


r2 = app.invoke(
    {"messages": [{"role": "user", "content": "What was my previous message?"}]},
    config=config,
)
print("AI:", r2["messages"][-1].content)

AI: Hello! I'm doing great, thank you for asking. 😊 

How are you doing today? What's on your mind—anything I can help you with?
AI: Your previous message was: **"Hello! How are you?"** 

Is there anything else you'd like to chat about or ask?


## 3) 打印消息历史

对话历史存放在 state 的 `messages` 里；同一个 `thread_id` 多次调用会把历史累积起来。

In [4]:
print("\nConversation History:")
for m in r2["messages"]:
    t = type(m).__name__
    content = getattr(m, "content", "")
    if isinstance(content, str) and len(content) > 220:
        content = content[:220] + " ..."
    print(f"{t}: {content!r}")


Conversation History:
HumanMessage: 'Hello! How are you?'
AIMessage: "Hello! I'm doing great, thank you for asking. 😊 \n\nHow are you doing today? What's on your mind—anything I can help you with?"
HumanMessage: 'What was my previous message?'
AIMessage: 'Your previous message was: **"Hello! How are you?"** \n\nIs there anything else you\'d like to chat about or ask?'


## 4) 多个 session：不同 `thread_id` = 不同会话（互相不串）

In [5]:
config_a = {"configurable": {"thread_id": "user_A"}}
config_b = {"configurable": {"thread_id": "user_B"}}

ra1 = app.invoke({"messages": [{"role": "user", "content": "My name is Alice."}]}, config=config_a)
rb1 = app.invoke({"messages": [{"role": "user", "content": "My name is Bob."}]}, config=config_b)

ra2 = app.invoke({"messages": [{"role": "user", "content": "What's my name?"}]}, config=config_a)
rb2 = app.invoke({"messages": [{"role": "user", "content": "What's my name?"}]}, config=config_b)

print("A:", ra2["messages"][-1].content)
print("B:", rb2["messages"][-1].content)

A: Your name is Alice! You told me at the very beginning of our conversation. 😊
B: Your name is Bob! You told me at the start of our conversation. 😊 

Is there anything else you'd like to talk about, Bob?
